# Perfilamiento complementario de la ECV 2025 a nivel departamental

La Primera Entrega perfiló la ECV a nivel de hogar (ver `hallazgos_perfilamiento.md` del repositorio de la Primera Entrega).
R6 y R7 integran la ECV con EVA **por departamento**, así que aquí se verifica lo que esa integración necesita:
códigos territoriales, tamaño de muestra, completitud FIES y prevalencia ponderada por departamento. Se observa **sin limpiar**.

In [1]:
# Configuracion. Los CSV crudos no estan en el repositorio: su ruta local se define
# por defecto en data/raw/ecv (ver data/raw/README.md); ECV_RAW_DIR en .env permite otra carpeta.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(RAIZ / '.env')
DIR_ECV = Path(os.getenv('ECV_RAW_DIR', RAIZ / 'data' / 'raw' / 'ecv'))
RUTA_VIVIENDA = DIR_ECV / 'Datos de la vivienda.csv'
RUTA_CONDICIONES = DIR_ECV / 'Condiciones de vida del hogar y tenencia de bienes.csv'
RUTA_DIVIPOLA = RAIZ / 'data' / 'reference' / 'divipola_departamentos.csv'
RUTA_EVA = RAIZ / 'data' / 'incoming' / 'profiling' / 'eva_2024_2025.json'
RUTA_R1_PRIMERA = RAIZ.parent / 'etl-inseguridad-alimentaria-colombia' / 'results' / 'R1_distribucion_territorial.csv'

FIES = [f'P3516S{i}' for i in range(1, 9)]
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 200)
print('Carpeta ECV configurada:', 'ECV_RAW_DIR' in os.environ, '| archivos encontrados:', RUTA_VIVIENDA.exists() and RUTA_CONDICIONES.exists())

Carpeta ECV configurada: False | archivos encontrados: True


In [2]:
# 1. Lectura con los parametros validados en la Primera Entrega (';', utf-8-sig,
# todo como texto). Solo las columnas que usa la integracion departamental.
vivienda = pd.read_csv(RUTA_VIVIENDA, sep=';', dtype=str, encoding='utf-8-sig',
                       usecols=['DIRECTORIO', 'P1_DEPARTAMENTO', 'P1_MUNICIPIO', 'REGION', 'CLASE', 'FEX_C'])
condiciones = pd.read_csv(RUTA_CONDICIONES, sep=';', dtype=str, encoding='utf-8-sig',
                          usecols=['DIRECTORIO', 'ORDEN', 'FEX_C', 'PROB_IAMG', 'PROB_IAG'] + FIES)
print('Vivienda:', vivienda.shape, '| DIRECTORIO unicos:', vivienda['DIRECTORIO'].nunique())
print('Condiciones:', condiciones.shape, '| (DIRECTORIO, ORDEN) duplicados:',
      condiciones.duplicated(['DIRECTORIO', 'ORDEN']).sum())

Vivienda: (86848, 6) | DIRECTORIO unicos: 86848
Condiciones: (87060, 13) | (DIRECTORIO, ORDEN) duplicados: 0


In [3]:
# 2. Integracion hogar -> vivienda por DIRECTORIO (regla de la Primera Entrega).
# El numero de hogares no debe cambiar.
hogares = condiciones.merge(vivienda, on='DIRECTORIO', how='left', suffixes=('', '_viv'), indicator=True)
print('Hogares antes:', len(condiciones), '| despues:', len(hogares))
print(hogares['_merge'].value_counts().to_dict())

Hogares antes: 87060 | despues: 87060
{'both': 87060, 'left_only': 0, 'right_only': 0}


In [4]:
# 3. Codigo de departamento: longitud observada y cruce con DIVIPOLA y con EVA.
print('Longitud de P1_DEPARTAMENTO:', hogares['P1_DEPARTAMENTO'].str.len().value_counts().to_dict())
print('Ejemplos de 1 caracter:', sorted(hogares.loc[hogares['P1_DEPARTAMENTO'].str.len() == 1, 'P1_DEPARTAMENTO'].unique()))
cod_ecv = set(hogares['P1_DEPARTAMENTO'].str.zfill(2))
divipola = pd.read_csv(RUTA_DIVIPOLA, dtype=str)
cod_div = set(divipola['Código Departamento'])
cod_eva = {r['c_digo_dane_departamento'] for r in json.loads(RUTA_EVA.read_text(encoding='utf-8'))['registros']}
print('Departamentos ECV (normalizados):', len(cod_ecv), '| DIVIPOLA:', len(cod_div), '| EVA:', len(cod_eva))
print('ECV sin DIVIPOLA:', sorted(cod_ecv - cod_div), '| DIVIPOLA sin ECV:', sorted(cod_div - cod_ecv))
print('ECV sin EVA:', sorted(cod_ecv - cod_eva), '| EVA sin ECV:', sorted(cod_eva - cod_ecv))

Longitud de P1_DEPARTAMENTO: {2: 80695, 1: 6365}
Ejemplos de 1 caracter: ['5', '8']


Departamentos ECV (normalizados): 33 | DIVIPOLA: 33 | EVA: 32
ECV sin DIVIPOLA: [] | DIVIPOLA sin ECV: []
ECV sin EVA: ['11'] | EVA sin ECV: []


In [5]:
# 4. Factor de expansion: formato, positividad y si coincide entre las dos tablas.
for c in ['FEX_C', 'FEX_C_viv']:
    s = hogares[c]
    print(c, '| nulos:', s.isna().sum(), '| con coma decimal:', s.str.contains(',', na=False).sum(),
          '| no numericos:', pd.to_numeric(s.str.replace(',', '.'), errors='coerce').isna().sum() - s.isna().sum())
fex = pd.to_numeric(hogares['FEX_C'].str.replace(',', '.'))
fex_viv = pd.to_numeric(hogares['FEX_C_viv'].str.replace(',', '.'))
print('FEX_C <= 0:', (fex <= 0).sum(), '| min:', fex.min(), '| max:', fex.max())
print('FEX_C hogar distinto de FEX_C vivienda:', (fex.round(6) != fex_viv.round(6)).sum())
print('Hogares expandidos (suma FEX_C):', round(fex.sum()))

FEX_C | nulos: 0 | con coma decimal: 87060 | no numericos: 0


FEX_C_viv | nulos: 0 | con coma decimal: 87060 | no numericos: 0
FEX_C <= 0: 0 | min: 1.545446408 | max: 4843.503512
FEX_C hogar distinto de FEX_C vivienda: 419
Hogares expandidos (suma FEX_C): 18952290


In [6]:
# 5. Medidas FIES: formato, rango (0-100), nulos y coherencia grave <= moderada o grave.
iamg = pd.to_numeric(hogares['PROB_IAMG'].str.replace(',', '.'), errors='coerce')
iag = pd.to_numeric(hogares['PROB_IAG'].str.replace(',', '.'), errors='coerce')
no_sabe = (hogares[FIES] == '3').any(axis=1)
print('PROB_IAMG nulos:', iamg.isna().sum(), '| PROB_IAG nulos:', iag.isna().sum())
print('Nulos FIES que coinciden con respuesta "no sabe" (3):', (iamg.isna() == no_sabe).all())
print('Fuera de 0-100 -> IAMG:', ((iamg < 0) | (iamg > 100)).sum(), '| IAG:', ((iag < 0) | (iag > 100)).sum())
print('IAG > IAMG (incoherente):', (iag > iamg + 1e-9).sum())
print(pd.DataFrame({'PROB_IAMG': iamg.describe(), 'PROB_IAG': iag.describe()}).round(3))

PROB_IAMG nulos: 718 | PROB_IAG nulos: 718
Nulos FIES que coinciden con respuesta "no sabe" (3): True
Fuera de 0-100 -> IAMG: 0 | IAG: 0
IAG > IAMG (incoherente): 0
       PROB_IAMG   PROB_IAG
count  86342.000  86342.000
mean      25.006      4.243
std       36.302     14.810
min        0.000      0.000
25%        0.000      0.000
50%        3.146      0.000
75%       37.506      0.002
max       99.825     75.223


In [7]:
# 6. Reconciliacion nacional con la cifra oficial del DANE (21,1 %), igual que en
# la Primera Entrega: SUM(FEX_C x PROB_IAMG) / SUM(FEX_C) sobre hogares con dato.
valido = iamg.notna()
print(f'Prevalencia moderada o grave: {(fex[valido] * iamg[valido]).sum() / fex[valido].sum():.2f} %')
print(f'Prevalencia grave:            {(fex[valido] * iag[valido]).sum() / fex[valido].sum():.2f} %')

Prevalencia moderada o grave: 21.10 %
Prevalencia grave:            3.42 %


In [8]:
# 7. Perfil por departamento: tamano de muestra, completitud FIES, peso poblacional
# y prevalencia ponderada (formula de D2).
hogares['cod_dep'] = hogares['P1_DEPARTAMENTO'].str.zfill(2)
t = pd.DataFrame({'cod_dep': hogares['cod_dep'], 'fex': fex, 'iamg': iamg, 'iag': iag, 'valido': valido})
t['fex_v'] = t['fex'].where(t['valido'], 0)
dep = t.groupby('cod_dep').agg(hogares_muestra=('fex', 'size'), hogares_sin_fies=('valido', lambda s: (~s).sum()),
                               hogares_expandidos=('fex', 'sum'), fex_valido=('fex_v', 'sum'))
dep['iamg_pond'] = t[t['valido']].assign(p=lambda d: d['fex'] * d['iamg']).groupby('cod_dep')['p'].sum() / dep['fex_valido']
dep['iag_pond'] = t[t['valido']].assign(p=lambda d: d['fex'] * d['iag']).groupby('cod_dep')['p'].sum() / dep['fex_valido']
dep['pct_sin_fies'] = (dep['hogares_sin_fies'] / dep['hogares_muestra'] * 100).round(2)
dep['pct_hogares_pais'] = (dep['hogares_expandidos'] / dep['hogares_expandidos'].sum() * 100).round(2)
dep = dep.join(divipola.set_index('Código Departamento')['Nombre Departamento'])
dep['en_eva'] = dep.index.isin(cod_eva)
dep = dep.round({'iamg_pond': 2, 'iag_pond': 2, 'hogares_expandidos': 0})
dep[['Nombre Departamento', 'hogares_muestra', 'hogares_sin_fies', 'pct_sin_fies', 'pct_hogares_pais',
     'iamg_pond', 'iag_pond', 'en_eva']].sort_values('hogares_muestra')

,Nombre Departamento,hogares_muestra,hogares_sin_fies,pct_sin_fies,pct_hogares_pais,iamg_pond,iag_pond,en_eva
cod_dep,,,,,,,,
88,"ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANT...",889,9,1.01,0.09,5.21,0.59,True
97,VAUPÉS,1532,21,1.37,0.08,22.31,1.86,True
91,AMAZONAS,1776,36,2.03,0.15,19.29,4.16,True
94,GUAINÍA,1797,44,2.45,0.08,22.63,3.32,True
99,VICHADA,2164,15,0.69,0.18,41.83,9.72,True
47,MAGDALENA,2269,12,0.53,2.39,36.11,7.41,True
27,CHOCÓ,2395,74,3.09,1.00,56.76,17.86,True
85,CASANARE,2421,22,0.91,0.89,17.50,1.16,True
95,GUAVIARE,2449,22,0.90,0.17,27.06,4.95,True


In [9]:
# 8. Contraste con el resultado R1 de la Primera Entrega (prevalencia por departamento).
# Debe dar lo mismo: es la misma fuente y la misma formula.
r1 = pd.read_csv(RUTA_R1_PRIMERA, encoding='utf-8-sig')
r1_dep = r1.groupby('departamento').apply(
    lambda g: (g['prevalencia_moderada_grave_pct'] * g['hogares']).sum() / g['hogares'].sum() if len(g) > 1
    else g['prevalencia_moderada_grave_pct'].iloc[0], include_groups=False)
print('Filas de R1 (region-departamento):', len(r1), '| departamentos:', r1['departamento'].nunique())
print('Departamentos con mas de una fila en R1:', r1['departamento'].value_counts().loc[lambda s: s > 1].to_dict())
comp = dep.assign(nombre=dep['Nombre Departamento'].str.upper()).set_index('nombre')['iamg_pond'].to_frame('notebook')
comp['primera_entrega'] = r1_dep
comp['diferencia'] = (comp['notebook'] - comp['primera_entrega']).round(2)
print('Departamentos sin pareja en R1:', comp['primera_entrega'].isna().sum())
print('Diferencia absoluta maxima:', comp['diferencia'].abs().max())
print('Departamentos con diferencia > 0,01 puntos:')
print(comp[comp['diferencia'].abs() > 0.01])
print('Nota: R1 trae Bogota en dos filas (dos regiones) y aqui se combinan ponderando por hogares de la')
print('muestra, no por factor de expansion; por eso Bogota puede diferir levemente. El resto debe coincidir.')

Filas de R1 (region-departamento): 34 | departamentos: 33
Departamentos con mas de una fila en R1: {'BOGOTÁ, D.C.': 2}
Departamentos sin pareja en R1: 0
Diferencia absoluta maxima: 0.17
Departamentos con diferencia > 0,01 puntos:
              notebook  primera_entrega  diferencia
nombre                                             
BOGOTÁ, D.C.      9.62         9.451163        0.17
Nota: R1 trae Bogota en dos filas (dos regiones) y aqui se combinan ponderando por hogares de la
muestra, no por factor de expansion; por eso Bogota puede diferir levemente. El resto debe coincidir.


In [10]:
# 9. Granularidad: la ECV trae municipio, pero su muestra no permite estimar a ese
# nivel. Se mide cuantos hogares hay por municipio para justificar el nivel departamental.
mun = hogares.groupby(['cod_dep', 'P1_MUNICIPIO']).size()
print('Municipios con hogares en la muestra:', len(mun))
print(mun.describe(percentiles=[.1, .25, .5, .75, .9]).round(1))
print('Municipios con menos de 30 hogares:', (mun < 30).sum())

Municipios con hogares en la muestra: 634
count     634.0
mean      137.3
std       220.6
min         7.0
10%        41.0
25%        58.0
50%        67.0
75%       108.8
90%       256.1
max      2947.0
dtype: float64
Municipios con menos de 30 hogares: 4


In [11]:
# 10. Peso de Bogota, que no tiene informacion agricola en EVA y quedara sin dato en R6 y R7.
bog = dep.loc['11']
print(f"Bogota: {bog['hogares_muestra']} hogares en muestra, {bog['pct_hogares_pais']} % de los hogares del pais, "
      f"prevalencia {bog['iamg_pond']} %")

Bogota: 2947 hogares en muestra, 16.77 % de los hogares del pais, prevalencia 9.62 %
